# CTHYB vs CTSEG vs CTINT: single-orbital spin-spin with $J_\perp(\tau)$

Same model as `spin_spin.py` / `ctseg_spin_spin.py` (U=4, $\beta$=10, bath from `ctint.ref.h5`):
$S_z S_z$ via `D0_tau` (Lang-Firsov in CTHYB) plus $J_\perp(\tau)$ via `Jperp_tau` (stochastic
`insert_dyn`/`remove_dyn` in CTHYB).

- `jperp_on = False` is the control: no stochastic vertices, everything goes through Lang-Firsov.
- `jperp_on = True` mixes Lang-Firsov $S_zS_z$ with stochastic $S^+S^-$. This is the case where the
  stochastic spin-flip operators need Lang-Firsov dressing.

Quantities: $G_\uparrow(\tau)$, $\langle S_z(\tau)S_z(0)\rangle$, and the $J_\perp$ perturbation-order
histogram (CTHYB `perturbation_order_dynamical` vs CTSEG `perturbation_order_J`, both count $S^+S^-$ pairs).

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from h5 import HDFArchive
from triqs.gfs import *
from triqs.plot.mpl_interface import *

%matplotlib inline
%config InlineBackend.figure_format = 'retina'
plt.rcParams['figure.dpi'] = 150
plt.rcParams['mathtext.fontset'] = 'cm'
plt.rcParams['mathtext.rm'] = 'serif'
plt.rc('font', size=12)

In [ ]:
data_loc = "/home/andrewhardy/Documents/Data/CTHYB_Data"
U, beta = 4.0, 10.0

J = "1.0"          # "0.25", "0.5", "1.0"
jperp_on = True    # False = Jperp-off control (Lang-Firsov only)

# CTHYB lf=True runs, per (J, jperp_on). The Jperp-on runs are the Aug 12-13 files.
cthyb_files = {
    ("1.0",  True):  "spin_spin_cthyb_J-1.0-U-4.0_1.0_b-10.0_nw-1000000_mins-22_lf=True_nl=22.h5",
    ("0.5",  True):  "spin_spin_cthyb_J-0.5-U-4.0_1.0_b-10.0_nw-1000000_mins-22_lf=True_nl=22.h5",
    ("0.25", True):  "spin_spin_cthyb_J-0.25-U-4.0_1.0_b-10.0_nw-1000000_mins-22_lf=True_nl=22.h5",
    ("1.0",  False): "spin_spin_cthyb_J-1.0-U-4.0_0.0_b-10.0_nw-1000000_mins-50_lf=True_nl=25.h5",
    ("0.5",  False): "spin_spin_cthyb_J-0.5-U-4.0_0.0_1.0_1.0_1.0_1.0_b-10.0_nw-1000000_mins-50_lf=True.h5",
}

def switches(on):
    return ("1.0" if on else "0.0") + "_1.0_1.0_1.0_1.0"

def files_for(J, on):
    f_hyb = cthyb_files.get((J, on))
    f_seg = f"spin_spin_ctseg_J-{J}-U-4.0_{switches(on)}_b-10.0.h5"
    f_int = f"spin_spin_ctint_J-{J}-U-4.0_{switches(on)}_b-10.0.h5"
    return [os.path.join(data_loc, f) if f and os.path.exists(os.path.join(data_loc, f)) else None
            for f in (f_hyb, f_seg, f_int)]

f_cthyb, f_ctseg, f_ctint = files_for(J, jperp_on)
print("CTHYB:", f_cthyb, "\nCTSEG:", f_ctseg, "\nCTINT:", f_ctint)

In [ ]:
def tau_of(g):
    return np.array([float(t) for t in g.mesh])

def order_mean(p):
    p = np.asarray(p, dtype=float)
    return (np.arange(len(p)) * p).sum() / p.sum()

def load_cthyb(f):
    with HDFArchive(f, "r") as A:
        O = A["O_tau"]
        return dict(G=A["G_tau"]["up"], tau_chi=tau_of(O), chi=O.data.real.copy(), sign=A["average_sign"],
                    pert_J=np.asarray(A["perturbation_order_dynamical"]) if "perturbation_order_dynamical" in A else None)

def load_ctseg(f):
    with HDFArchive(f, "r") as A:
        nn = A["nn_tau"]
        chi = 0.25 * (nn['up', 'up'].data[:, 0, 0] + nn['down', 'down'].data[:, 0, 0]
                      - nn['up', 'down'].data[:, 0, 0] - nn['down', 'up'].data[:, 0, 0]).real
        return dict(G=A["G_tau"]["up"], tau_chi=tau_of(nn['up', 'up']), chi=chi, sign=A["average_sign"],
                    pert_J=np.asarray(A["perturbation_order_J"]) if "perturbation_order_J" in A else None)

def load_ctint(f, n_tau=2001):
    # chiAB_tau lives on a DLR imaginary-time mesh (30 nodes): go through the DLR coefficients
    # to a uniform tau grid instead of reading the raw (non-uniform) nodes.
    with HDFArchive(f, "r") as A:
        chi = make_gf_imtime(make_gf_dlr(A["chiAB_tau"]), n_tau)
        return dict(G=None, tau_chi=tau_of(chi), chi=chi.data[:, 0].real.copy(), sign=None, pert_J=None)

res = {}
if f_cthyb: res["CTHYB"] = load_cthyb(f_cthyb)
if f_ctseg: res["CTSEG"] = load_ctseg(f_ctseg)
if f_ctint: res["CTINT"] = load_ctint(f_ctint)

In [ ]:
print(f"J = {J}, Jperp {'ON' if jperp_on else 'OFF'}")
for name, r in res.items():
    chi_half = np.interp(beta / 2, r["tau_chi"], r["chi"])
    line = f"{name:6s} chi(0) = {r['chi'][0]:.4f}   chi(beta/2) = {chi_half:.4f}"
    if r["sign"] is not None: line += f"   sign = {r['sign']:.4f}"
    if r["pert_J"] is not None: line += f"   <k_Jperp> = {order_mean(r['pert_J']):.3f}"
    print(line)

In [ ]:
plt.figure(figsize=(8, 6))
style = {"CTHYB": dict(color="orange", linestyle="-"), "CTSEG": dict(color="blue", linestyle="--"),
         "CTINT": dict(color="green", linestyle=":")}
for name, r in res.items():
    if r["G"] is not None:
        plt.plot(tau_of(r["G"]), r["G"].data[:, 0, 0].real, linewidth=2.0, label=name, **style[name])
plt.title(r"$G_\uparrow(\tau)$ Comparison" + f", J={J}, " + (r"$J_\perp$ on" if jperp_on else r"$J_\perp$ off"))
plt.xlabel(r"$\tau$")
plt.ylabel(r"$G(\tau)$")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
for name, r in res.items():
    plt.plot(r["tau_chi"], r["chi"], linewidth=2.0, label=name, **style[name])
plt.title(r"$\langle S_z(\tau) S_z(0) \rangle$ Comparison" + f", J={J}, " + (r"$J_\perp$ on" if jperp_on else r"$J_\perp$ off"))
plt.xlabel(r"$\tau$")
plt.ylabel(r"$\langle S_z(\tau) S_z(0) \rangle$")
plt.legend()
plt.show()

In [ ]:
# Difference to CTSEG on CTSEG's tau mesh
if "CTSEG" in res:
    ref = res["CTSEG"]
    plt.figure(figsize=(8, 4))
    for name, r in res.items():
        if name == "CTSEG": continue
        plt.plot(ref["tau_chi"], np.interp(ref["tau_chi"], r["tau_chi"], r["chi"]) - ref["chi"],
                 linewidth=1.5, label=f"{name} - CTSEG", color=style[name]["color"])
    plt.axhline(0, color="k", linewidth=0.8)
    plt.xlabel(r"$\tau$")
    plt.ylabel(r"$\Delta\langle S_z(\tau) S_z(0) \rangle$")
    plt.legend()
    plt.show()

In [ ]:
# Jperp perturbation order: both histograms count S+S- pairs; normalized to 1
if jperp_on:
    plt.figure(figsize=(8, 4))
    for name, r in res.items():
        if r["pert_J"] is None: continue
        p = np.asarray(r["pert_J"], dtype=float)
        p = p / p.sum()
        k_max = min(len(p), 16)
        plt.plot(np.arange(k_max), p[:k_max], "-o", label=f"{name}  <k> = {order_mean(r['pert_J']):.3f}", color=style[name]["color"])
    plt.xlabel(r"$J_\perp$ perturbation order $k$")
    plt.ylabel(r"$P(k)$")
    plt.legend()
    plt.show()

## Scan over J: $\langle S_z(\beta/2) S_z(0)\rangle$, Jperp on vs off

In [ ]:
J_values = ["0.25", "0.5", "1.0"]
loaders = {"CTHYB": load_cthyb, "CTSEG": load_ctseg, "CTINT": load_ctint}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, on in zip(axes, [False, True]):
    for idx, name in enumerate(["CTHYB", "CTSEG", "CTINT"]):
        xs, ys = [], []
        for Jv in J_values:
            f = files_for(Jv, on)[idx]
            if f is None: continue
            r = loaders[name](f)
            xs.append(float(Jv))
            ys.append(np.interp(beta / 2, r["tau_chi"], r["chi"]))
        if xs:
            ax.plot(xs, ys, "o", color=style[name]["color"], markersize=9 - 2 * idx, label=name,
                    linestyle=style[name]["linestyle"])
    ax.set_title(r"$J_\perp$ on" if on else r"$J_\perp$ off (control)")
    ax.set_xlabel(r"$J$")
axes[0].set_ylabel(r"$\langle S_z(\beta/2) S_z(0) \rangle$")
axes[0].legend()
plt.show()